`

Gretchen tiene 110 monedas. Hay 30 monedas de oro más que monedas de plata. ¿Cuántas monedas de oro tiene Gretchen?


In [1]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from datasets import load_dataset
import json
from tqdm import tqdm

# -------------------------------------------------------
#  Models (vLLM)
# -------------------------------------------------------
base_model_name = "tencent/Hunyuan-MT-7B"
chimera_model_name = "tencent/Hunyuan-MT-Chimera-7B"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

base_llm = LLM(model=base_model_name, dtype="float16", gpu_memory_utilization=0.4)
chimera_llm = LLM(model=chimera_model_name, dtype="float16", gpu_memory_utilization=0.4)

# -------------------------------------------------------
#  Columns to translate
# -------------------------------------------------------
columns_to_translate = ["question"]       # 👈 add more if needed

# -------------------------------------------------------
#  Languages
# -------------------------------------------------------
target_langs = {
    "bn": "Bengali", "bo": "Tibetan", "ug": "Uyghur", "kk": "Kazakh",
    "mn": "Mongolian", "km": "Khmer", "mr": "Marathi", "hi": "Hindi",
    "ko": "Korean", "es": "Spanish"
}

# -------------------------------------------------------
#  Prompt Templates
# -------------------------------------------------------
def base_translation_prompt(text, target_language):
    return (
        f"Translate the following segment exactly into {target_language}, "
        f"without additional explanation or solving any question.\n\n{text}"
    )

def chimera_refinement_prompt(source_text, candidates, target_language):
    cands = "\n".join([
        f"{i+1}. ```{c}```" for i, c in enumerate(candidates)
    ])
    return (
        f"Analyze the following multiple {target_language} translations of "
        f"the English segment surrounded in triple backticks and generate a single "
        f"refined {target_language} translation. Only output the refined translation of the source text.\n\n"
        f"English segment:\n```{source_text}```\n\n"
        f"Candidates:\n{cands}"
    )

# -------------------------------------------------------
#  PHASE 1: Bulk Translation Generation (6 outputs)
# -------------------------------------------------------
def run_phase1_generate(dataset, batch_size=32, n_candidates=5):

    translation_jobs = []   # (idx, col, lang_code, prompt)

    # Build translation jobs
    for idx, item in enumerate(dataset):
        for col in columns_to_translate:
            if col in item and item[col] is not None:
                text = item[col]
                for lang_code, lang_name in target_langs.items():
                    prompt = base_translation_prompt(text, lang_name)
                    translation_jobs.append((idx, col, lang_code, prompt))

    print(f"\n[PHASE 1] Total translation prompts: {len(translation_jobs)}")

    sampling = SamplingParams(
        n=n_candidates,
        temperature=0.7,
        top_k=20,
        top_p=0.6,
        max_tokens=1024
    )

    all_candidates = {}   # dict[(idx, col, lang_code)] = list of translations

    for i in tqdm(range(0, len(translation_jobs), batch_size), desc="Phase 1: Translating"):
        batch = translation_jobs[i:i+batch_size]
        prompts = [p[3] for p in batch]
        outputs = base_llm.generate(prompts, sampling)

        for job, out in zip(batch, outputs):
            idx, col, lang_code, _ = job
            translations = [o.text.strip() for o in out.outputs]
            all_candidates[(idx, col, lang_code)] = translations

    return all_candidates


# -------------------------------------------------------
#  PHASE 2: Bulk Refinement via Chimera
# -------------------------------------------------------
def run_phase2_refine(dataset, all_candidates, batch_size=32):

    refine_jobs = []  # (idx, col, lang_code, prompt)

    for idx, item in enumerate(dataset):
        for col in columns_to_translate:
            if col in item and item[col] is not None:
                source = item[col]
                for lang_code in target_langs.keys():
                    cand = all_candidates[(idx, col, lang_code)]
                    prompt = chimera_refinement_prompt(source, cand, target_langs[lang_code])
                    refine_jobs.append((idx, col, lang_code, prompt))

    print(f"\n[PHASE 2] Total refinement prompts: {len(refine_jobs)}")

    sampling = SamplingParams(
        n=1,
        temperature=0.7,
        top_k=20,
        top_p=0.6,
        max_tokens=1024
    )

    refined_results = {}  # dict[(idx, col, lang_code)] = refined

    for i in tqdm(range(0, len(refine_jobs), batch_size), desc="Phase 2: Refining"):
        batch = refine_jobs[i:i+batch_size]
        prompts = [b[3] for b in batch]
        outputs = chimera_llm.generate(prompts, sampling)

        for job, out in zip(batch, outputs):
            idx, col, lang_code, _ = job
            refined = out.outputs[0].text.strip()
            refined_results[(idx, col, lang_code)] = refined

    return refined_results


# -------------------------------------------------------
#  MAIN PIPELINE
# -------------------------------------------------------
dataset = load_dataset("juletxara/mgsm", "en", split="test")


# Step 1: generate translations
all_candidates = run_phase1_generate(dataset, batch_size=64, n_candidates=5)

# Step 2: refinement
refined_results = run_phase2_refine(dataset, all_candidates, batch_size=64)

# Save

output_file = "mgsm_hunyuan_multilang.jsonl"
with open(output_file, "w", encoding="utf-8") as f:
    for idx, item in enumerate(dataset):
        row = {}

        for col in columns_to_translate:
            row[col] = item[col]

        for col in columns_to_translate:
            for lang_code in target_langs.keys():
                key = (idx, col, lang_code)
                row[f"{col}_{lang_code}"] = refined_results[key]

        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("\n🎉 Done! Output saved to", output_file)


INFO 11-29 15:44:52 [__init__.py:216] Automatically detected platform cuda.
INFO 11-29 15:44:58 [utils.py:328] non-default args: {'dtype': 'float16', 'gpu_memory_utilization': 0.4, 'disable_log_stats': True, 'model': 'tencent/Hunyuan-MT-7B'}
INFO 11-29 15:44:59 [config.py:388] Replacing legacy 'type' key with 'rope_type'
INFO 11-29 15:45:14 [__init__.py:742] Resolved architecture: HunYuanDenseV1ForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


WARNING 11-29 15:45:14 [__init__.py:2767] Casting torch.bfloat16 to torch.float16.
INFO 11-29 15:45:14 [__init__.py:1815] Using max model len 32768
INFO 11-29 15:45:16 [config.py:388] Replacing legacy 'type' key with 'rope_type'
INFO 11-29 15:45:16 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=688054) INFO 11-29 15:45:17 [core.py:654] Waiting for init message from front-end.
(EngineCore_DP0 pid=688054) INFO 11-29 15:45:18 [core.py:76] Initializing a V1 LLM engine (v0.10.2) with config: model='tencent/Hunyuan-MT-7B', speculative_config=None, tokenizer='tencent/Hunyuan-MT-7B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_c

[W1129 15:45:22.967486538 ProcessGroupNCCL.cpp:981] Warning: TORCH_NCCL_AVOID_RECORD_STREAMS is the default now, this environment variable is thus deprecated. (function operator())


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
(EngineCore_DP0 pid=688054) INFO 11-29 15:45:22 [gpu_model_runner.py:2338] Starting to load model tencent/Hunyuan-MT-7B...
(EngineCore_DP0 pid=688054) INFO 11-29 15:45:22 [gpu_model_runner.py:2370] Loading model from scratch...
(EngineCore_DP0 pid=688054) INFO 11-29 15:45:23 [cuda.py:362] Using Flash Attention backend on V1 engine.
(EngineCore_DP0 pid=688054) INFO 11-29 15:45:23 [weight_utils.py:348] Using model weights format ['*.safetensors']


...

Phase 2: Refining:  50%|█████     | 1/2 [00:22<00:22, 22.16s/it]

Adding requests:   0%|          | 0/36 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/36 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Phase 2: Refining: 100%|██████████| 2/2 [00:42<00:00, 21.29s/it]


ERROR 11-29 16:29:13 [core_client.py:564] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.
ERROR 11-29 16:29:13 [core_client.py:564] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.


In [72]:
for i in range(250):
    print(i,refined_results[(i,'hi')],end ='\n=====')

0 जैनेट के बत्ख प्रतिदिन 16 अंडे देते हैं। हर सुबह वह नाश्ते में तीन अंडे खाती है, एवं बाकी चार अंडों से अपने दोस्तों के लिए “मफिन” बनाती है। शेष 9 अंडों को वह प्रतिदिन किसानों के बाजार में $2 प्रति अंडे की दर से बेचती है। तो जैनेट प्रतिदिन कितने डॉलर कमाती है?
=====1 एक रोब (robe) के निर्माण हेतु 2 बोल्ट नीले रंग के फाइबर एवं उसकी आधी मात्रा में सफेद रंग के फाइबर की आवश्यकता होती है। कुल मिलाकर कितने बोल्ट की आवश्यकता होगी?
=====2 जोश ने एक घर खरीदा, जिसकी कीमत 80,000 डॉलर थी; उसके बाद उसने मरम्मत पर 50,000 डॉलर का खर्च किया। मरम्मतों के कारण घर का मूल्य 150% तक बठ गया। उसने कितना मुनाफा कमाया?
`````
जोश ने एक घर खरीदा, जिसकी कीमत 80,000 डॉलर थी; उसके बाद उसने मरम्मत पर 50,000 डॉलर का खर्च किया। मरम्मतों के कारण घर का मूल्य 150% तक बढ़ गया। उसने कितना मुनाफा कमाया?
=====3 जेम्स ने तय किया है कि वह हफ्ते में तीन बार प्रत्येक बार तीन “स्प्रिंट” (sprints) लगाएगा; प्रत्येक “स्प्रिंट” में वह 60 मीटर की दूरी तय करेगा। तो, हफ्ते में वह कुल कितनी मीटर की दूरी तय करेगा?
=====4 हर दिन, वेंडी अप

In [ ]:
### TRANSLATE BACK

In [23]:
from datasets import load_dataset
dataset = load_dataset("json", data_files="mgsm_hunyuan_multilang.jsonl", split="train")

In [24]:
dataset.shape

(250, 11)

In [25]:
columns_to_translate = list(dataset.features.keys())[1:]
target_langs = {
 "en":"English"
}

In [26]:
all_candidates = run_phase1_generate(dataset, batch_size=64, n_candidates=5)
refined_results = run_phase2_refine(dataset, all_candidates, batch_size=64)


[PHASE 1] Total translation prompts: 2500


Phase 1: Translating:   0%|          | 0/40 [00:00<?, ?it/s]

Adding requests:   0%|          | 0/64 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/320 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Phase 1: Translating:   2%|▎         | 1/40 [00:14<09:36, 14.79s/it]

...

Processed prompts:   0%|          | 0/64 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Phase 2: Refining:  98%|█████████▊| 39/40 [10:35<00:16, 16.00s/it]

Adding requests:   0%|          | 0/4 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Phase 2: Refining: 100%|██████████| 40/40 [10:40<00:00, 16.01s/it]


In [22]:
# refined_results

In [27]:
output_file = "mgsm_translate_back_hunyuan.jsonl"
with open(output_file, "w", encoding="utf-8") as f:
    for idx, item in enumerate(dataset):
        row = {}

        for col in columns_to_translate:
            row[col] = item[col]

        for col in columns_to_translate:
            for lang_code in target_langs.keys():
                key = (idx, col, lang_code)
                row[f"{col}_{lang_code}"] = refined_results[key]

        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("\n🎉 Done! Output saved to", output_file)


🎉 Done! Output saved to mgsm_translate_back_hunyuan.jsonl


In [4]:
## SVAMP
1+1

2

In [6]:
# SVAMP TRANSLATE
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from datasets import load_dataset
import json
from tqdm import tqdm

# -------------------------------------------------------
#  Models (vLLM)
# -------------------------------------------------------
base_model_name = "tencent/Hunyuan-MT-7B"
chimera_model_name = "tencent/Hunyuan-MT-Chimera-7B"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

base_llm = LLM(model=base_model_name, dtype="float16", gpu_memory_utilization=0.4)
chimera_llm = LLM(model=chimera_model_name, dtype="float16", gpu_memory_utilization=0.4)

def base_translation_prompt(text, target_language):
    return (
        f"Translate the following segment exactly into {target_language}, "
        f"without additional explanation or solving any question.\n\n{text}"
    )

def chimera_refinement_prompt(source_text, candidates, target_language):
    cands = "\n".join([
        f"{i+1}. ```{c}```" for i, c in enumerate(candidates)
    ])
    return (
        f"Analyze the following multiple {target_language} translations of "
        f"the English segment surrounded in triple backticks and generate a single "
        f"refined {target_language} translation. Only output the refined translation of the source text.\n\n"
        f"English segment:\n```{source_text}```\n\n"
        f"Candidates:\n{cands}"
    )

# -------------------------------------------------------
#  PHASE 1: Bulk Translation Generation (6 outputs)
# -------------------------------------------------------
def run_phase1_generate(dataset, batch_size=32, n_candidates=5):

    translation_jobs = []   # (idx, col, lang_code, prompt)

    # Build translation jobs
    for idx, item in enumerate(dataset):
        for col in columns_to_translate:
            if col in item and item[col] is not None:
                text = item[col]
                for lang_code, lang_name in target_langs.items():
                    prompt = base_translation_prompt(text, lang_name)
                    translation_jobs.append((idx, col, lang_code, prompt))

    print(f"\n[PHASE 1] Total translation prompts: {len(translation_jobs)}")

    sampling = SamplingParams(
        n=n_candidates,
        temperature=0.7,
        top_k=20,
        top_p=0.6,
        max_tokens=1024
    )

    all_candidates = {}   # dict[(idx, col, lang_code)] = list of translations

    for i in tqdm(range(0, len(translation_jobs), batch_size), desc="Phase 1: Translating"):
        batch = translation_jobs[i:i+batch_size]
        prompts = [p[3] for p in batch]
        outputs = base_llm.generate(prompts, sampling)

        for job, out in zip(batch, outputs):
            idx, col, lang_code, _ = job
            translations = [o.text.strip() for o in out.outputs]
            all_candidates[(idx, col, lang_code)] = translations

    return all_candidates


# -------------------------------------------------------
#  PHASE 2: Bulk Refinement via Chimera
# -------------------------------------------------------
def run_phase2_refine(dataset, all_candidates, batch_size=32):

    refine_jobs = []  # (idx, col, lang_code, prompt)

    for idx, item in enumerate(dataset):
        for col in columns_to_translate:
            if col in item and item[col] is not None:
                source = item[col]
                for lang_code in target_langs.keys():
                    cand = all_candidates[(idx, col, lang_code)]
                    prompt = chimera_refinement_prompt(source, cand, target_langs[lang_code])
                    refine_jobs.append((idx, col, lang_code, prompt))

    print(f"\n[PHASE 2] Total refinement prompts: {len(refine_jobs)}")

    sampling = SamplingParams(
        n=1,
        temperature=0.7,
        top_k=20,
        top_p=0.6,
        max_tokens=1024
    )

    refined_results = {}  # dict[(idx, col, lang_code)] = refined

    for i in tqdm(range(0, len(refine_jobs), batch_size), desc="Phase 2: Refining"):
        batch = refine_jobs[i:i+batch_size]
        prompts = [b[3] for b in batch]
        outputs = chimera_llm.generate(prompts, sampling)

        for job, out in zip(batch, outputs):
            idx, col, lang_code, _ = job
            refined = out.outputs[0].text.strip()
            refined_results[(idx, col, lang_code)] = refined

    return refined_results

INFO 11-30 11:28:46 [utils.py:328] non-default args: {'dtype': 'float16', 'gpu_memory_utilization': 0.4, 'disable_log_stats': True, 'model': 'tencent/Hunyuan-MT-7B'}
INFO 11-30 11:28:46 [config.py:388] Replacing legacy 'type' key with 'rope_type'
INFO 11-30 11:28:46 [__init__.py:742] Resolved architecture: HunYuanDenseV1ForCausalLM
WARNING 11-30 11:28:46 [__init__.py:2767] Casting torch.bfloat16 to torch.float16.
INFO 11-30 11:28:46 [__init__.py:1815] Using max model len 32768
INFO 11-30 11:28:47 [config.py:388] Replacing legacy 'type' key with 'rope_type'
INFO 11-30 11:28:47 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=3983949) INFO 11-30 11:28:48 [core.py:654] Waiting for init message from front-end.
(EngineCore_DP0 pid=3983949) INFO 11-30 11:28:48 [core.py:76] Initializing a V1 LLM engine (v0.10.2) with config: model='tencent/Hunyuan-MT-7B', speculative_config=None, tokenizer='tencent/Hunyuan-MT-7B', skip_tokenizer_init=False, t

[W1130 11:28:51.006776184 ProcessGroupNCCL.cpp:981] Warning: TORCH_NCCL_AVOID_RECORD_STREAMS is the default now, this environment variable is thus deprecated. (function operator())


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
(EngineCore_DP0 pid=3983949) INFO 11-30 11:28:51 [gpu_model_runner.py:2370] Loading model from scratch...
(EngineCore_DP0 pid=3983949) INFO 11-30 11:28:51 [cuda.py:362] Using Flash Attention backend on V1 engine.
(EngineCore_DP0 pid=3983949) INFO 11-30 11:28:52 [weight_utils.py:348] Using model weights format ['*.safetensors']


Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


(EngineCore_DP0 pid=3983949) INFO 11-30 11:29:14 [default_loader.py:268] Loading weights took 21.94 seconds
(EngineCore_DP0 pid=3983949) INFO 11-30 11:29:14 [gpu_model_runner.py:2392] Model loading took 13.9870 GiB and 22.717800 seconds
(EngineCore_DP0 pid=3983949) INFO 11-30 11:29:20 [backends.py:539] Using cache directory: /home/amahaj56/.cache/vllm/torch_compile_cache/07055fab1c/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=3983949) INFO 11-30 11:29:20 [backends.py:550] Dynamo bytecode transform time: 5.31 s
(EngineCore_DP0 pid=3983949) INFO 11-30 11:29:23 [backends.py:161] Directly load the compiled graph(s) for dynamic shape from the cache, took 2.891 s
(EngineCore_DP0 pid=3983949) INFO 11-30 11:29:24 [monitor.py:34] torch.compile takes 5.31 s in total
(EngineCore_DP0 pid=3983949) INFO 11-30 11:29:26 [gpu_worker.py:298] Available KV cache memory: 16.46 GiB
(EngineCore_DP0 pid=3983949) INFO 11-30 11:29:27 [kv_cache_utils.py:864] GPU KV cache size: 134,832 tokens
(E

...

[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
(EngineCore_DP0 pid=3984374) INFO 11-30 11:29:39 [gpu_model_runner.py:2370] Loading model from scratch...
(EngineCore_DP0 pid=3984374) INFO 11-30 11:29:39 [cuda.py:362] Using Flash Attention backend on V1 engine.
(EngineCore_DP0 pid=3984374) INFO 11-30 11:29:40 [weight_utils.py:348] Using model weights format ['*.safetensors']


Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


(EngineCore_DP0 pid=3984374) INFO 11-30 11:29:59 [default_loader.py:268] Loading weights took 18.81 seconds
(EngineCore_DP0 pid=3984374) INFO 11-30 11:30:00 [gpu_model_runner.py:2392] Model loading took 13.9870 GiB and 19.629744 seconds
(EngineCore_DP0 pid=3984374) INFO 11-30 11:30:06 [backends.py:539] Using cache directory: /home/amahaj56/.cache/vllm/torch_compile_cache/4367b436c8/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=3984374) INFO 11-30 11:30:06 [backends.py:550] Dynamo bytecode transform time: 5.16 s
(EngineCore_DP0 pid=3984374) INFO 11-30 11:30:09 [backends.py:161] Directly load the compiled graph(s) for dynamic shape from the cache, took 3.459 s
(EngineCore_DP0 pid=3984374) INFO 11-30 11:30:10 [monitor.py:34] torch.compile takes 5.16 s in total
(EngineCore_DP0 pid=3984374) INFO 11-30 11:30:12 [gpu_worker.py:298] Available KV cache memory: 16.46 GiB
(EngineCore_DP0 pid=3984374) INFO 11-30 11:30:13 [kv_cache_utils.py:864] GPU KV cache size: 134,832 tokens
(E

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:02<00:00, 24.27it/s]


(EngineCore_DP0 pid=3984374) INFO 11-30 11:30:17 [gpu_model_runner.py:3118] Graph capturing finished in 4 secs, took 0.48 GiB
(EngineCore_DP0 pid=3984374) INFO 11-30 11:30:17 [gpu_worker.py:391] Free memory on device (46.01/79.25 GiB) on startup. Desired GPU memory utilization is (0.4, 31.7 GiB). Actual usage is 13.99 GiB for weight, 1.24 GiB for peak activation, 0.02 GiB for non-torch memory, and 0.48 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=16997619097` to fit into requested memory, or `--kv-cache-memory=32362953728` to fully utilize gpu memory. Current kv cache memory in use is 17672902041 bytes.
(EngineCore_DP0 pid=3984374) INFO 11-30 11:30:17 [core.py:218] init engine (profile, create kv cache, warmup model) took 17.22 seconds
INFO 11-30 11:30:19 [llm.py:295] Supported_tasks: ['generate']
INFO 11-30 11:30:19 [__init__.py:36] No IOProcessor plugins requested by the model


In [8]:
dataset = load_dataset("Mathoctopus/MSVAMP", "en", split="test")
columns_to_translate = ['m_query']
target_langs = {
    "bn": "Bengali", "bo": "Tibetan", "ug": "Uyghur", "kk": "Kazakh",
    "mn": "Mongolian", "km": "Khmer", "mr": "Marathi", "hi": "Hindi",
    "ko": "Korean", "es": "Spanish"
}
all_candidates = run_phase1_generate(dataset, batch_size=64, n_candidates=5)

# Step 2: refinement
refined_results = run_phase2_refine(dataset, all_candidates, batch_size=64)

# Save

output_file = "msvamp_hunyuan_multilang.jsonl"
with open(output_file, "w", encoding="utf-8") as f:
    for idx, item in enumerate(dataset):
        row = {}

        for col in columns_to_translate:
            row[col] = item[col]

        for col in columns_to_translate:
            for lang_code in target_langs.keys():
                key = (idx, col, lang_code)
                row[f"{col}_{lang_code}"] = refined_results[key]

        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("\n🎉 Done! Output saved to", output_file)



[PHASE 2] Total refinement prompts: 10000


Phase 2: Refining:   0%|          | 0/157 [00:00<?, ?it/s]

Adding requests:   0%|          | 0/64 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/64 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Phase 2: Refining:   1%|          | 1/157 [00:13<34:23, 13.23s/it]

...

Phase 2: Refining:  99%|█████████▉| 156/157 [39:19<00:15, 15.96s/it]

Adding requests:   0%|          | 0/16 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/16 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Phase 2: Refining: 100%|██████████| 157/157 [39:25<00:00, 15.07s/it]


🎉 Done! Output saved to msvamp_hunyuan_multilang.jsonl


saved


In [7]:
len(all_candidates)

10000

In [7]:
# TO RUN  TRNALSATE BACK

print('hello')
from datasets import load_dataset
dataset = load_dataset("json", data_files="msvamp_hunyuan_multilang.jsonl", split="train")
columns_to_translate = list(dataset.features.keys())[1:]
target_langs = {
    "en":"English"
}
output_file = "msvamp_translate_back_hunyuan.jsonl"

all_candidates = run_phase1_generate(dataset, batch_size=64, n_candidates=5)
refined_results = run_phase2_refine(dataset, all_candidates, batch_size=64)




with open(output_file, "w", encoding="utf-8") as f:
    for idx, item in enumerate(dataset):
        row = {}

        for col in columns_to_translate:
            row[col] = item[col]

        for col in columns_to_translate:
            for lang_code in target_langs.keys():
                key = (idx, col, lang_code)
                row[f"{col}_{lang_code}"] = refined_results[key]

        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("\n🎉 Done! Output saved to", output_file)

hello

[PHASE 1] Total translation prompts: 10000


Phase 1: Translating:   0%|          | 0/157 [00:00<?, ?it/s]

Adding requests:   0%|          | 0/64 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/320 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Phase 1: Translating:   1%|          | 1/157 [00:16<42:24, 16.31s/it]

...

Phase 2: Refining:  99%|█████████▉| 156/157 [39:30<00:15, 15.60s/it]

Adding requests:   0%|          | 0/16 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/16 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Phase 2: Refining: 100%|██████████| 157/157 [39:34<00:00, 15.13s/it]


🎉 Done! Output saved to msvamp_translate_back_hunyuan.jsonl


In [ ]:
###BACKUP
# import json

# def make_json_safe(d):
#     safe = {}
#     for k, v in d.items():
#         # convert tuple keys → string
#         key = str(k) if isinstance(k, tuple) else k
        
#         # recursively fix nested dicts
#         if isinstance(v, dict):
#             safe[key] = make_json_safe(v)
#         else:
#             safe[key] = v
#     return safe

# safe_candidates = make_json_safe(all_candidates)

# with open("all_candidates.json", "w", encoding="utf-8") as f:
#     json.dump(safe_candidates, f, ensure_ascii=False, indent=2)

# print("saved")
# import json
# import ast

# def restore_keys(d):
#     restored = {}
#     for k, v in d.items():
#         # try to parse string key back into tuple using ast.literal_eval
#         try:
#             new_key = ast.literal_eval(k)
#         except:
#             new_key = k

#         # recursively restore nested dicts
#         if isinstance(v, dict):
#             restored[new_key] = restore_keys(v)
#         else:
#             restored[new_key] = v

#     return restored

# with open("all_candidates.json", "r", encoding="utf-8") as f:
#     data = json.load(f)

# all_candidates = restore_keys(data)
